In [0]:
1. Column Level redaction 
2. Row Level Security
3. Data Masking

In [0]:
create or replace table dev.naval_silver.sales_customers as select * from samples.bakehouse.sales_customers

In [0]:
grant use catalog on catalog dev to `account users`;
grant use schema on schema dev.naval_silver to `account users`;
grant select on table dev.naval_silver.sales_customers to `account users`

In [0]:
select * from dev.naval_silver.sales_customers

Column Level

In [0]:
grant select on table dev.naval_gold.customers to `da`;
grant select on table dev.naval_gold.customers to `dataeng`;
grant select on table dev.naval_gold.customers_geo to `dataeng`

In [0]:
create or replace view dev.naval_gold.customers as
select 
customerID, 
first_name,
last_name,
CASE WHEN
    is_account_group_member('dataeng') THEN 'REDACTED'
    ELSE email_address
  END AS email_address,
phone_number,
CASE WHEN
    is_account_group_member('da') THEN 'REDACTED'
    ELSE address
  END AS address,
city,
state,
country,
continent,
postal_zip_code,
gender 
from dev.naval_silver.sales_customers

In [0]:
create or replace view dev.naval_gold.customers as
select 
customerID, 
first_name,
last_name,
CASE WHEN
    is_account_group_member('dataeng') THEN email_address
    ELSE 'REDACTED' 
  END AS email_address,
phone_number,
CASE WHEN
    is_account_group_member('da') THEN 'REDACTED'
    ELSE address
  END AS address,
city,
state,
country,
continent,
postal_zip_code,
gender 
from dev.naval_silver.sales_customers

In [0]:
select * from dev.naval_gold.customers

Row Level

In [0]:
--create or replace view dev.naval_gold.customers as
select 
customerID, 
first_name,
last_name,
email_address,
phone_number,
address,
city,
state,
country,
continent,
postal_zip_code,
gender 
from dev.naval_silver.sales_customers

In [0]:
create or replace view dev.naval_gold.customers_geo as 
select 
customerID, 
first_name,
last_name,
email_address,
phone_number,
address,
city,
state,
country,
continent,
postal_zip_code,
gender 
from dev.naval_silver.sales_customers
where CASE
    WHEN is_account_group_member('dataeng') THEN country ='Australia'
    ELSE True
  END;

In [0]:
create or replace view dev.naval_gold.customers_geo as 
select 
customerID, 
first_name,
last_name,
email_address,
phone_number,
address,
city,
state,
country,
continent,
postal_zip_code,
gender 
from dev.naval_silver.sales_customers
where CASE
    WHEN is_account_group_member('dataeng') THEN country = 'Australia'
    WHEN is_account_group_member('da') THEN country = 'Japan'
    WHEN is_account_group_member('account users') THEN country = 'USA'
    ELSE True
  END;

In [0]:
select * from dev.naval_gold.customers_geo

Data Masking 

In [0]:
CREATE OR REPLACE FUNCTION dev.naval_silver.datamask(x STRING)
  RETURNS STRING
  RETURN CONCAT(REPEAT("*", LENGTH(x) - 2), RIGHT(x, 2)
); 

In [0]:
create or replace view dev.naval_gold.customers_m as
select 
customerID, 
first_name,
last_name,
CASE WHEN
    is_account_group_member('dataeng') THEN email_address
    ELSE dev.naval_silver.datamask(email_address)
  END AS email_address,
phone_number,
CASE WHEN
    is_account_group_member('da') THEN 'REDACTED'
    ELSE address
  END AS address,
city,
state,
country,
continent,
postal_zip_code,
gender 
from dev.naval_silver.sales_customers

In [0]:
select * from dev.naval_gold.customers_m